In [ ]:
# -*- coding: utf-8 -*-
'''
Titel: _script_

file: script (functions)
type: python (jupyter notebook python)
coding: utf-8

Functions:
    number: Confirmation the script is connected to _main_
    clip_3band_orto
    create_tiles 
    train_model
    detect_cars
    cleanning_function
    network_lines
    polygon_iou
    compute_iou_matrix
    match_predictions
    compute_metrics
    mean_iou_of_matches
    compute_ap

written by: Elina N.A

'''

In [1]:
def number(num):
    """
    Return a confirmation string indicating that the script is connected
    to the program's main execution block.

    Parameters
    ----------
    num : str
        An unused input parameter. It is accepted for interface
        compatibility but does not affect the function's behavior.

    Returns
    -------
    str
        The string "Yes!", indicating that the script is connected
        to the `__main__` execution context.

    """

    # String which ensures script is connected to _main_
    connected = "Yes!"

    return connected

# Clip big ortos and change to 4-band to 3-band
Because pretrained-model doesn't take 4 bands

In [2]:
def clip_3band_orto(big_orto, clip_bounds, clip_raster):
    """
    Clip a 4-band ortho image to a vector boundary and export a 3-band RGB
    GeoTIFF. 

    Parameters
    ----------
    big_orto : str and path to orto tif-file
        Path to the input orthoimage. Must contain at least three bands
        corresponding to red, green, and blue.
    clip_bounds : str and path to vector shp-file.
        Path to a vector file (e.g., GeoPackage, Shapefile) containing one
        or more polygon geometries used as clipping boundaries.
    clip_raster : str and path to be orto tif-file.
        Output path for the clipped RGB GeoTIFF.

    Returns
    -------
    None
        The function writes the clipped raster to disk and does not return
        any objects.

    Notes
    -----
    - The function reads only the first three bands of the input raster.
    - Clipping is performed using `rasterio.mask.mask`, which expects GeoJSON-like geometry dictionaries extracted via Fiona.
    - A `MemoryFile` is used to avoid writing an intermediate RGB raster to disk before clipping.
    - Metadata from the original raster is copied and updated to reflect the new dimensions, transform, and band count.
    - The output raster is always written as a 3-band uint8 GeoTIFF.

    Examples
    --------
    >>> clip_3band_orto(
    ...     big_orto="big_test_orto.tif",
    ...     clip_bounds="clip_outline.gpkg",
    ...     clip_raster="clip_train_raster.tif"
    ... )
    """
    
    # Open outline vector file to use GeoJSON-like geometry
    with fiona.open(clip_bounds, "r") as f:              # AI-presented: Get the outline shape to use for clip
        shapes = [feature["geometry"] for feature in f]

    # Open big orto to be able to clip and create RGB-band
    with rasterio.open(big_orto) as img:
    
        # Read red, green and blue band from big orto
        red_band = img.read(1)
        green_band = img.read(2)
        blue_band = img.read(3)

        # Stack RGB-bands into a single array 
        rgb = np.stack([red_band, green_band, blue_band]).clip(0, 255)  # Ensure dtype is unit8
    
    mem_rgb = rasterio.io.MemoryFile()     # AI-presented when I wanted to clip orto and update meta band count
                                                    # without writing new file inbetween
    # Open memory file
    with mem_rgb.open(                  # AI-presented and helped as above explination
            driver= "GTiff",
            dtype="uint8",
            count=3,
            height=rgb.shape[1],  # Same height as RGB stack
            width=rgb.shape[2],   # Same width as RGB stack
            transform=img.transform,   # Same transform as big orto
            crs=img.crs               # Same coordinate system as big orto
    ) as mem:                             
        mem.write(rgb[0], 1)      # AI-presented and helped as above explination
        mem.write(rgb[1], 2)
        mem.write(rgb[2], 3)

        # Clip memory RGB file to vector outline shape
        out_image, out_transform = rasterio.mask.mask(mem, shapes, crop=True)

        # Copy big meta. 
        out_meta = img.meta.copy()

        # Update new meta for clip orto
        out_meta.update(
            {
                "driver": "GTiff",
                "dtype": "uint8",
                "height": out_image.shape[1],   # Same height as clip orto
                "width": out_image.shape[2],   # Same width as clip orto
                "transform": out_transform,   # Same transform as clip orto
                "count":3,                   # Update band count (from 4 to 3)
            }
        )

    # Terminate the MemoryFile
    mem_rgb.close()    

    # Open and write new clipped RGB raster with new meta
    with rasterio.open(clip_raster, "w", **out_meta) as dst:
        dst.write(out_image)
        

# Generate tiles for training (fine-tune model)

In [3]:
def create_tiles(in_raster, tile_folder, in_polygon):
    """
    Generate training tiles with `geoai.export_geotiff_tiles` from an input raster and polygon annotation shapefile.

    Parameters
    ----------
    in_raster : str and path to orto tif-file
        Path to the input raster from which tiles will be generated.
    tile_folder : str and path to be tile-folder
        Directory where the output tiles and metadata will be written.
        The folder will be created if it does not already exist.
    in_polygon : str and path to vector shp-file
        Path to a vector file containing polygon annotations. The attribute
        field `"Classname"` is used to assign class values to each tile.

    Returns
    -------
    None
        The function writes tiles and metadata to disk but does not return
        any objects.

    Notes
    -----
    - Tile size is fixed at 256×256 pixels.
    - Stride is fixed at 64 pixels; creates an overlap of 75% between tiles.
    - Only tiles containing annotated objects are exported with `skip_empty_tiles=True`.
    - Metadata is written in `RCNN_Masks` format, compatible with Mask R‑CNN training pipelines.
    - The function also generates an overview image of the tiling layout with `create_overview=True`.

    Examples
    --------
    >>> create_tiles(
    ...     in_raster="clip_train_orto.tif",
    ...     tile_folder="training_tiles/",
    ...     in_polygon="train_polygon.shp"
    ... )
    """

    # Export geotiff tiles as trainingdata 
    geoai.export_geotiff_tiles(
        in_raster=in_raster,
        out_folder=tile_folder,
        in_class_data=in_polygon, 
        tile_size=256,
        stride=64,
        buffer_radius=0,
        skip_empty_tiles=True,
        metadata_format='RCNN_Masks',
        class_value_field="Classname",
        create_overview=True
    )
    

# Fine-tune model (train)

In [4]:
def train_model(tile_folder, out_model_folder, trained_model):
    """
    Train a Mask R-CNN model with `geoai.train_MaskRCNN_model` using GeoAI tile datasets.

    Parameters
    ----------
    tile_folder : str and path to tile-folder
        Path to the folder containing training tiles. 
        Must include `images/` and `labels/` subdirectories.
    out_model_folder : str and path to folder
        Directory where the trained model and training logs will be saved.
    trained_model : str and path to pretrained model-file
        Path to a pretrained Mask R-CNN model to use for fine‑tuning. 
        If `resume_training=True`, training continues from this checkpoint.

    Returns
    -------
    None
        The function trains a model and writes outputs to disk but does not
        return any objects.

    Notes
    -----
    - Uses 3‑channel RGB input with `num_channels=3`).
    - Batch size is fixed at 4; depends on if GPU memory is limited.
    - Training runs for 10 epochs with a learning rate of 0.005.
    - Validation split is fixed at 10% of the dataset.
    - `pretrained=True` enables transfer learning from the provided model.
    - `resume_training=True` continues training from the checkpoint at `trained_model`.

    Examples
    --------
    >>> train_model(
    ...     tile_folder="training_tiles/",
    ...     out_model_folder="models/best_model.pth",    # To be the new DL-model
    ...     trained_model="car_detection_usa.pth"
    ... )
    """

    # Train a MaskRCNN model 
    geoai.train_MaskRCNN_model(
        images_dir=fr"{tile_folder}/images",
        labels_dir=fr"{tile_folder}/labels",
        output_dir=fr"{model_folder}",
        num_channels=3,
        batch_size=4,
        num_epochs=10,
        learning_rate=0.005,
        val_split=0.1,
        pretrained=True,
        pretrained_model_path=trained_model,
        resume_training=True
    )


# Inference

In [5]:
def detect_cars(test_raster, predicted_raster, use_model_folder):
    """
    Run object detection with `geoai.object_detection` on an input raster using a trained Mask R-CNN model.

    Parameters
    ----------
    test_raster : str and path to tif-file
        Path to the input raster on which object detection will be performed.
        Must be a 3‑band RGB GeoTIFF or compatible format.
    predicted_raster : str and path to tif-file
        Output path where the prediction raster will be written. The file
        will contain detected object masks or bounding boxes depending on
        the model configuration.
    use_model_folder : str and path to folder
        Directory containing the trained model. The function expects the
        file `best_model.pth` to exist inside this folder.

    Returns
    -------
    None
        The function writes the prediction raster to disk but does not
        return any objects.

    Notes
    -----
    - Inference uses a sliding‑window approach with a window size of 256
      pixels and an overlap of 64 pixels.
    - Confidence threshold is fixed at 0.5; detections below this value
      are discarded.
    - Batch size is set to 4; adjust if GPU memory is limited.
    - The model must have been trained with 3‑channel RGB input.

    Examples
    --------
    >>> detect_cars(
    ...     test_raster="ortho_test_rgb.tif",
    ...     predicted_raster="predictions.tif",
    ...     use_model_folder="models/run01/"
    ... )
    """

    # Use object detection as inference type
    geoai.object_detection(
        input_path=test_raster,
        output_path=predicted_raster,
        model_path=fr"{use_model_folder}/best_model.pth",
        window_size=256,
        overlap=64,
        confidence_threshold=0.5,
        batch_size=4,
        num_channels=3,
    )


This function is designed for post‑processing instance segmentation predictions. It removes overly large or small polygons, splits merged predictions using reference geometry, and ensures all geometries are valid, simple, and within expected size thresholds.

In [6]:
def cleanning_function(gdf_pred, true_ref, clean_polygon_path):
    """
    Clean predicted polygons by intersecting them with ground truth data,
    exploding multipart geometries, simplifying shapes, computing geometric properties, 
    filtering by size/shape constraints, and exporting the cleaned polygons to a GeoPackage.

    Parameters
    ----------
    gdf_pred : geopandas.GeoDataFrame
        GeoDataFrame containing predicted polygons (e.g., from Mask R-CNN). Must be in SWEREF 3006 (EPSG:3006) or a CRS compatible with `true_ref`.
    true_ref : geopandas.GeoDataFrame
        Ground‑truth reference polygons used to split or refine predicted geometries. Must share the same CRS as `gdf_pred`.
    clean_polygon_path : str or pathlib.Path
        Output path for the cleaned polygons. A GeoPackage (.gpkg) will be written with layer name `'clean'`.

    Returns
    -------
    None
        The function writes the cleaned polygons to disk but does not return
        any objects.

    Notes
    -----
    - `gpd.overlay(..., how="intersection")` is used to break apart large predicted polygons by intersecting them with ground truth geometry.
    - Multipart geometries are exploded into individual polygons.
    - Polygons are simplified with a tolerance of 0.2 m while preserving topology.
    - `geoai.add_geometric_properties` computes area, perimeter, major/minor axis lengths, and other shape metrics.
    - Filtering thresholds:
        * area_m2 > 8
        * area_m2 < 60
        * minor_length_m > 1
      These thresholds remove noise, tiny fragments, and overly large shapes.
    - Output is saved as a GeoPackage.

    Examples
    --------
    >>> cleanning_function(
    ...     gdf_pred=predicted_gdf,
    ...     true_ref=reference_gdf,
    ...     clean_polygon_path="cleaned_predictions.gpkg"
    ... )
    """

    # Select inside predicted which overlay ground truth data to seperate big polygons
    inside = gpd.overlay(gdf_pred, gdf_ref, how="intersection")

    # Explode multipolygons
    exploded = inside.explode(index_parts=True)

    # Create a list with only the geometry column
    locations = [i for i in exploded['geometry']]
    
    # Create GeoDataFrame with geometry being locations-list 
    gdf_loc = gpd.GeoDataFrame(geometry=locations).set_crs(3006)  # Set coordinate system

    # Simplify polygons with 0.2 meter and preserving topology
    gdf_loc.simplify(.2, preserve_topology=True)

    # Add geometric properties with geoai
    gdf_loc = geoai.add_geometric_properties(gdf_loc)

    # Filter out to small and too big polygons
    gdf_filter = gdf_loc[(gdf_loc["area_m2"] > 8) & (gdf_loc["area_m2"] < 60) & (gdf_loc["minor_length_m"] > 1)]

    # Save clean predicted polygons
    gdf_filter.to_file(clean_polygon_path, driver='GPKG', layer='clean')
    

In [7]:
def network_lines(clean_predicted_gdf):
    """
    Select two random predicted car polygons, compute their centroids, and 
    generate a drivable street‑network graph around a fixed reference point (Karlstad, Sweden). 
    
    The function returns the origin and destination centroids along with a projected network graph suitable for routing.

    Parameters
    ----------
    clean_predicted_gdf : geopandas.GeoDataFrame
        GeoDataFrame containing cleaned predicted polygons (e.g., car detections). 
        Must contain a valid geometry column in EPSG:3006.

    Returns
    -------
    origin_center : shapely.geometry.Point
        Centroid of the randomly selected origin polygon.
    destination_center : shapely.geometry.Point
        Centroid of the randomly selected destination polygon.
    G_projected_3006 : networkx.MultiDiGraph
        Drivable street‑network graph generated by OSMnx and projected to SWEREF 99 TM (EPSG:3006).

    Notes
    -----
    - Two random rows are selected from the GeoDataFrame to represent origin and destination vehicles.
    - Centroids are used instead of polygon geometries to simplify routing.
    - The network is generated using:
        * a fixed geocoded center point ("Vasagatan, Karlstad, Sweden")
        * a search radius of 900 meters
        * `network_type="drive"` to restrict edges to drivable roads
    - The graph is projected to EPSG:3006 using `ox.project_graph`.
    - The function prints the selected row indices and centroid coordinates for debugging and reproducibility.

    """
    
    # Generate a random integer from total numbers of rows in gdf
    random_number_o = random.randint(0, len(clean_predicted_gdf))
    random_number_d = random.randint(0, len(clean_predicted_gdf))

    # Locate the random row number in gdf for origin and destination cars
    origin = clean_predicted_gdf.geometry.iloc[random_number_o]
    destination = clean_predicted_gdf.geometry.iloc[random_number_d]

    # Get centroid of origin and destination car polygons
    origin_center = origin.centroid
    destination_center = destination.centroid  

    # Set center point string of network
    karlstad = "Vasagatan, Karlstad, Sweden"

    # Geocode center point string 
    center_point = ox.geocode(karlstad)

    # Set distance
    dist = 900

    # Create network graph from point with set distance
    G = ox.graph_from_point(center_point, dist=dist, network_type="drive")   # Netork-type ensures it's netork lines where cars drives
    G_projected_3006 = ox.project_graph(G, to_crs='EPSG:3006')             # Project graph to SWEREF99 TM

    print(random_number_o, " ", random_number_d) # Output: A random number between 10 and 50
    print(f'd_lant: {origin_center} och d_lon: {destination_center}.')

    # Returns origon, destination point and network graph
    return origin_center, destination_center, G_projected_3006
    

# Compute Accuracy Assesment

AI, Copilot, has been used to help create functions. It's important that the logic is correct when computing accuracy for object detection, that which is to be compared to another GIS-application like ArcGIS Pro 'Compute Accuracy for Object Detection.' 

I believe there is some function in geoai-library thich creates confusion matrix and the other parameters, but this made me understand the logic behind the equation much better.  

In [8]:
def polygon_iou(p, r):
    """
    Compute the Intersection over Union (IoU) between two polygons.

    Parameters
    ----------
    p : shapely.geometry.Polygon or shapely.geometry.MultiPolygon
        Predicted polygon geometry.
    r : shapely.geometry.Polygon or shapely.geometry.MultiPolygon
        Reference (ground‑truth) polygon geometry.

    Returns
    -------
    float
        The IoU value in the range [0, 1]. Returns 0.0 if the union area
        is zero or if the polygons do not overlap.

    Notes
    -----
    - IoU is computed as:
                ``IoU = intersection_area / union_area``

    - If the polygons do not intersect, the intersection area is zero.
    - If either geometry is empty or invalid, the IoU may be zero or undefined depending on the input.
    - This function assumes both geometries are in the same coordinate reference system.

    """

    # Get areas of predicted intersection referencedata
    inter = p.intersection(r).area

    # Get area of predicted union with referencedata (ground‑truth)
    union = p.union(r).area

    # Calculate iou as intersection 0ver union
    iou = inter / union if union > 0.0 else 0.0   # If union is 0.0 IoU becomes 0.0
                                             # AI-presented the single line with if-sats which protect from division with zero

    # Return iou
    return iou

In [9]:
def compute_iou_matrix(preds, refs):
    """
    Compute a pairwise Intersection over Union (IoU) matrix between predicted
    and reference polygons.
    
    The result is a matrix of shape (N_pred, N_ref), where each entry
    M[i, j] contains the IoU between prediction i and reference j.

    Parameters
    ----------
    preds : list of shapely.geometry.Polygon or GeoSeries
        Iterable containing predicted polygon geometries.
    refs : list of shapely.geometry.Polygon or GeoSeries
        Iterable containing reference (ground‑truth) polygon geometries.

    Returns
    -------
    numpy.ndarray
        A 2D array of shape (len(preds), len(refs)) where each element is-
        the IoU value between a predicted and reference polygon.

    Notes
    -----
    - IoU is computed using the helper function `polygon_iou`.
    - The function assumes all geometries are valid and in the same CRS.
    - This matrix is typically used for:
        * greedy matching
        * Hungarian matching
        * TP/FP/FN computation
        * precision/recall/AP evaluation
    - If either list is empty, the resulting matrix will have zero rows
      or zero columns.

    """

    # Creates 2D-matris with 0
    M = np.zeros((len(preds), len(refs)))

    # Compare each predicted geometry with all reference geometries
    for pred_idx, pred_row in enumerate(preds):

        for ref_idx, ref_row in enumerate(refs):

            # Calculate intersection of union to place in matris on predicted and reference index
            M[pred_idx, ref_idx] = polygon_iou(pred_row, ref_row)

    # Return matris
    return M

In [10]:
def match_predictions(M, iou_thresh=0.5):     # AI-helped with outline of the whole function
                                                  # double checked with 'Compute Accuracy for OB' in ArcGIS Pro
    
    """
    Perform greedy one‑to‑one matching between predicted and reference
    polygons using an IoU matrix, and compute TP, FP, FN, precision, and
    recall at each prediction step.

    This function implements a confidence‑agnostic greedy matching strategy:
    for each predicted polygon (row of the IoU matrix), the reference polygon
    with the highest IoU is selected. If the IoU exceeds the threshold and
    the reference polygon has not been matched before, the prediction is
    counted as a true positive (TP). Otherwise, it is counted as a false
    positive (FP). False negatives (FN) are computed based on unmatched
    reference polygons.

    Parameters
    ----------
    M : numpy.ndarray
        A 2D IoU matrix of shape (N_pred, N_ref), where M[i, j] is the IoU between predicted polygon i and reference polygon j.
    iou_thresh : float, optional
        Minimum IoU required to consider a prediction a true positive. Defaults to 0.5.

    Returns
    -------
    matches : list of tuple
        List of matched index pairs `(pred_idx, ref_idx)` for all true
        positives.
    tp : int
        Total number of true positives.
    fp : int
        Total number of false positives.
    fn : int
        Total number of false negatives.
    metrics : dict
        Dictionary containing the final precision and recall values.
    precisions : list of float
        Precision values computed after each prediction step.
    recalls : list of float
        Recall values computed after each prediction step.

    Notes
    -----
    - Matching is greedy and does not use the Hungarian algorithm.
    - Each reference polygon can be matched at most once.
    - Precision and recall are computed incrementally after each prediction,
      enabling precision–recall curve construction.
    - False negatives are computed as:
    
              ``FN = N_ref - number_of_matched_references``

    - This function assumes predictions are already sorted in the desired
      evaluation order (e.g., by confidence score).

    Examples
    --------
    >>> matches, tp, fp, fn, metrics, precisions, recalls = match_predictions(M, iou_thresh=0.5)
    
    """

    # Get number predicted and reference objects from IoU matrix shape
    num_preds, num_refs = M.shape

    # Keep track of which reference polygons have already been matched
    matched_refs = set()       # AI-presented 

    # Store values
    ious_tp = []     # IoU-values for true positive (tp)
    matches = []     # Matched index pairs between pred_idx and ref_idx

    # Counters for tp and fp (false positive)
    tp = 0
    fp = 0

    # Empty lists to store 
    precisions = []
    recalls = []

    # Loop through each predicted polygon in iou-matrix
    for i in range(num_preds):

        # Find the reference polygon with highest iou for this prediction
        j = np.argmax(M[i])

        # Extract the iou value for this best match
        iou_val = M[i, j]

        # Check if iou is above the threshold (0.5) and reference polygon is not already matched (e.g. set())
        if M[i, j] >= iou_thresh and j not in matched_refs:

            # Count as true positive
            tp += 1

            # Mark this reference polygon as matched
            matched_refs.add(j)

            # Store iou value for tp
            ious_tp.append(iou_val)

            # Store the matched pair (pred = i, ref = j)
            matches.append((i, j))
            
        else:
            # Else count this prediction is fp
            fp += 1

        # False negative (fn) is what remains of reference object after subtracting the matched polygons
        fn = num_refs - len(matched_refs)

        # Compute precision and recall with script function below
        metrics = compute_metrics(tp, fp, fn)

        # Extract precision and recall values
        for key, value in metrics.items():
            
            if key == "Precision":

                # Store precision value (for AP)
                precisions.append(value)
    
            if key == "Recall":

                # Store recall value (for AP)
                recalls.append(value)

    # Return 
    return matches, tp, fp, fn, metrics, precisions, recalls

In [11]:
def compute_metrics(tp, fp, fn):

    """
    Compute precision, recall, and F1-score from counts of true positives,
    false positives, and false negatives.

    Parameters
    ----------
    tp : int
        Number of true positives.
    fp : int
        Number of false positives.
    fn : int
        Number of false negatives.

    Returns
    -------
    dict
        A dictionary containing:
        - "Precision": float in the range [0, 1].
        - "Recall": float in the range [0, 1].
        - "F1 Score": float in the range [0, 1].

    Notes
    -----
    - precision is defined as:
          ``precision = TP / (TP + FP)``

    - recall is defined as:
          ``recall = TP / (TP + FN)``

    - f1 is defined as the harmonic mean of precision and recall:
          ``F1 = 2 * (precision * recall) / (precision + recall)``

    - All metrics return 0 if their denominator is zero.
    - This function is typically called repeatedly during greedy matching
      to build precision–recall curves.

    """

    # Compute precision
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0  # Avoide division by zero

    # Compute recall
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0    # Avoide division by zero

    # Compute F1-score
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0 
                                                                                 # Avoide division by zero

    # Return all three metrics in a dict
    return {
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
    }

In [12]:
def mean_iou_of_matches(M, tp_indices):
    """
    Compute the mean Intersection over Union (IoU) across all matched
    prediction–reference pairs.

    This function takes an IoU matrix and a list of true‑positive index
    pairs and returns the average IoU for those matches. It is typically
    used to summarize the quality of correctly detected objects in an
    object‑detection or instance‑segmentation evaluation pipeline.

    Parameters
    ----------
    M : numpy.ndarray
        A 2D IoU matrix of shape (N_pred, N_ref), where M[i, j] contains
        the IoU between predicted polygon i and reference polygon j.
    tp_indices : list of tuple
        List of matched index pairs `(pred_idx, ref_idx)` corresponding to tp. 
        Each tuple identifies one IoU value in the matrix.

    Returns
    -------
    float
        The mean IoU across all matched pairs. Returns 0 if no matches
        are provided.

    Notes
    -----
    - This function only considers true‑positive matches; false positives and 
      false negatives do not contribute to the mean IoU.
    - If `tp_indices` is empty, the function returns 0 to avoid division by zero.
    - This metric is useful for evaluating how well the model localizes objects it successfully detects.
      
    """

    # If-sats where if no to matches return 0
    if len(tp_indices) == 0:
        return 0

    # Compute mean of iou for all matched predicted (i) and reference (j) pairs.
    iou_mean = np.mean([M[i, j] for i, j in tp_indices])

    # Return
    return iou_mean

In [13]:
def compute_ap(precisions, recalls):
    """
    Compute Average Precision (AP) using the 11‑point interpolation method
    from the Pascal VOC 2007 evaluation protocol.

    This function takes lists of precision and recall values (typically
    generated incrementally during greedy matching) and computes AP by
    sampling recall thresholds from 0.0 to 1.0 in 11 steps. For each
    threshold, the maximum precision observed at any recall ≥ threshold
    is used. The final AP is the mean of these 11 precision values.

    Parameters
    ----------
    precisions : list of float
        Precision values computed at each prediction step.
    recalls : list of float
        Recall values computed at each prediction step. Must be aligned
        with `precisions` in order and length.

    Returns
    -------
    float
        The interpolated Average Precision (AP) score in the range [0, 1].

    Notes
    -----
    - This function implements the classic Pascal VOC 2007 AP metric:
      
      *11 recall thresholds:*  
      ``t ∈ {0.0, 0.1, 0.2, ..., 1.0}``

      *For each threshold:*  
      ``p(t) = max precision where recall ≥ t``

      *Final AP:*  
      ``AP = mean(p(t) for all thresholds)``

    - Precision and recall values are sorted by recall before interpolation.
    - This method is less strict than COCO‑style AP@[0.5:0.95], but is
      widely used for simpler evaluation pipelines.

    """

    # Sort by recall: ascending list
    order = np.argsort(recalls)

    # Reorder the recall values according to sorted indices
    recalls = np.array(recalls)[order]   

    # Reorder the precision values using to sorted indices
    precisions = np.array(precisions)[order]

    # Set count int
    ap = 0             # Average precision (ap) is area under precision-recall curve
    
    # 11-point interpolation (Pascal VOC style)

    # Loop over 11 recall thresholds (Pascal VOC style)
    for t in np.linspace(0, 1, 11):                     # AI-presented: np.linespace(0, 1, 11)
    
        # For this threshold t:
        # Find the maximum precision among all points where recall >= t
        # If no recall values reach t, use precision = 0
        p = precisions[recalls >= t].max() if np.any(recalls >= t) else 0
        
        # Add precision value to AP-total
        ap += p

    # Return AP as the average of 11 interpolated precision values 
    return ap/11